# Break Kempe's four-colour step on Heawood's map

*Adversarial Mathematics From First Principles*, [chapter 7](https://programmer.ie/books/math/07-chapter/) · [worked example](https://programmer.ie/examples/math/07-chapter/)

Run the cells in order. Everything below the program is taken from the laboratory's recorded, hash-verified run; the program is a small, self-contained demonstration of the same mathematics.

Kempe's 1879 proof of the four-colour theorem was accepted for eleven years,
until Heawood published a map on which one step fails: two Kempe-chain swaps
made together. The map here is the project's transcription of Heawood's
Fig. 18, 25 regions and 69 shared boundaries. The program checks each step of
Heawood's account by exhaustive search.

## Run it

Standard library only; nothing to install.

In [1]:
"""Kempe's double swap on Heawood's 1890 map (Fig. 18), by exhaustive computation.

The map is the project's transcription of Heawood's plate: 25 regions, 69 shared
boundaries. Colours: r red, b blue, y yellow, g green; X is the uncoloured region.
Standard library only.
"""
COLOUR = dict(
    Gout="g", Bout="b", Yout="y", Y1="y", B1="b", R1="r", G1="g", Y2="y", G2="g", R2="r",
    B2="b", R3="r", B3="b", Y3="y", X=None, R4="r", G3="g", B4="b", G4="g", Y4="y",
    R5="r", G5="g", Y5="y", R6="r", B5="b",
)
HEAWOOD = {1: "R3", 2: "B3", 3: "R4", 4: "G3", 5: "Y3"}   # X's neighbours, in order round X
BOUNDARIES = """B1-G1 B1-Gout B1-R1 B1-R2 B1-Y1 B2-R2 B2-R3 B3-Gout B3-R4 B3-Y1 B3-Yout B4-G4
B4-R5 B4-Y5 B5-Yout Bout-G3 Bout-Gout Bout-Y3 Bout-Yout G1-B2 G1-R1 G1-R2 G1-Y2 G2-B3 G2-R3
G2-Y1 G3-B4 G3-R5 G3-Y4 G4-B5 G4-Y5 G4-Yout G5-B5 G5-Y5 Gout-Yout R1-G2 R1-Y1 R1-Y2 R2-Gout
R3-B3 R3-X R4-B4 R4-G3 R4-G4 R4-Yout R5-G5 R5-Y5 R6-B5 R6-Bout R6-G5 R6-Y4 R6-Yout X-B3 X-G3
X-R4 X-Y3 Y1-Gout Y2-B2 Y2-G2 Y2-R3 Y3-B2 Y3-G3 Y3-Gout Y3-R2 Y3-R3 Y4-Bout Y4-G5 Y4-R5
Y5-B5""".split()


def neighbours(drop=()):
    adj = {v: set() for v in COLOUR}
    for e in BOUNDARIES:
        a, b = e.split("-")
        if e not in drop:
            adj[a].add(b)
            adj[b].add(a)
    return adj


def chain(adj, col, start, c1, c2):
    """The Kempe chain: the connected two-coloured region containing start."""
    seen, stack = {start}, [start]
    while stack:
        for w in adj[stack.pop()]:
            if col.get(w) in (c1, c2) and w not in seen:
                seen.add(w)
                stack.append(w)
    return seen


def swap(col, part, c1, c2):
    return {v: (c2 if c == c1 else c1) if v in part else c for v, c in col.items()}


def clashes(adj, col):
    return sorted({tuple(sorted((a, b))) for a in adj for b in adj[a] if col[a] and col[a] == col[b]})


def free(adj, col):
    return sorted(set("rbyg") - {col[v] for v in adj["X"]})


def four_colourable(adj):
    order, col = sorted(adj, key=lambda v: -len(adj[v])), {}

    def go(i):
        if i == len(order):
            return True
        for c in "rbyg":
            if all(col.get(w) != c for w in adj[order[i]]):
                col[order[i]] = c
                if go(i + 1):
                    return True
                del col[order[i]]
        return False

    return go(0)


def analyse(drop=()):
    adj, n, col = neighbours(drop), HEAWOOD, COLOUR
    rg = chain(adj, col, n[1], "r", "g")
    ry = chain(adj, col, n[3], "r", "y")
    both = swap(swap(col, rg, "r", "g"), ry, "r", "y")
    steps = {
        "printed colouring is proper": clashes(adj, col) == [],
        "X's neighbours are r, b, r, g, y": [col[n[k]] for k in range(1, 6)] == list("rbrgy"),
        "2 and 5 in one blue-yellow chain": n[5] in chain(adj, col, n[2], "b", "y"),
        "2 and 4 in one blue-green chain": n[4] in chain(adj, col, n[2], "b", "g"),
        "1's red-green chain avoids 4": n[4] not in rg,
        "3's red-yellow chain avoids 5": n[5] not in ry,
        "red-green swap alone frees nothing": free(adj, swap(col, rg, "r", "g")) == [],
        "red-yellow swap alone frees nothing": free(adj, swap(col, ry, "r", "y")) == [],
        "both swaps together clash": clashes(adj, both) != [],
        "the map is four-colourable anyway": four_colourable(adj),
    }
    return steps, clashes(adj, both)


print(f"{len(COLOUR)} regions, {len(BOUNDARIES)} boundaries")
steps, clash = analyse()
for name, ok in steps.items():
    print(f"  {'yes' if ok else 'NO ':3}  {name}")
print(f"after both swaps, neighbouring regions with the same colour: {clash}")
print(f"Kempe's step fails as Heawood says: {all(steps.values())}")
print()
# Failure control: remove the one boundary Heawood's premise depends on.
steps, _ = analyse(drop=("B3-Gout",))
print(f"control, without the B3-Gout boundary: Heawood's account holds? {all(steps.values())}")

25 regions, 69 boundaries
  yes  printed colouring is proper
  yes  X's neighbours are r, b, r, g, y
  yes  2 and 5 in one blue-yellow chain
  yes  2 and 4 in one blue-green chain
  yes  1's red-green chain avoids 4
  yes  3's red-yellow chain avoids 5
  yes  red-green swap alone frees nothing
  yes  red-yellow swap alone frees nothing
  yes  both swaps together clash
  yes  the map is four-colourable anyway
after both swaps, neighbouring regions with the same colour: [('Gout', 'Yout')]
Kempe's step fails as Heawood says: True

control, without the B3-Gout boundary: Heawood's account holds? False


## What the output shows

Each swap is safe on its own and neither frees a colour; made together they
give two neighbouring regions, Gout and Yout, the same colour. Yet the map is
still four-colourable, so it refutes Kempe's *argument*, not the theorem. The
control shows the result depends on the reading of the plate: remove one
boundary and Heawood's account no longer holds. That is why the transcription's
fidelity is the open question.

## What the laboratory checked

The recorded run `20261113T233009Z-4b4a1765` ([evidence](https://github.com/ernanhughes/math/tree/main/experiments/investigations/p06-kempe/runner-runs/2026-11-13-pinned)) is the book's result; the program above is a reader-sized demonstration of the same mathematics. **Evidence:** Exact computation on a transcribed historical graph; no proof kernel. **Status:** `computed_correspondence_pending`.

> On the graph of Heawood's Fig. 18, with Heawood's printed colouring, Kempe's five-neighbour step fails as Heawood says: each of the two Kempe-chain swaps is safe alone, neither frees a colour, and made together they give two neighbouring regions the same colour; the map is nevertheless four-colourable.

| Check | System | Recorded outcome |
| --- | --- | --- |
| `inputs` | Pinned inputs | all inputs match their pins |
| `compute` | Python (graph search on the transcribed map) | exit 0, verdict kempe_step_fails_as_heawood_says |

Failure controls: **1 of 1** rejected or refused as required (`drop-B3-Gout`).
Correspondence `P06-CORR-01` (does the formal statement say what the claim means?): **pending the author's approval**.
Contract `P06-KEMPE-HEAWOOD` version 2, sha256 `095c94b9c4104084…`; `result.json` sha256 `c73111f9c81e3830…`.

## Not shown

As the laboratory's own record states:

- That the transcription is Heawood's map: that awaits a human reading of the scan.
- What Kempe himself wrote: his 1879 paper has not been transcribed.
- Anything kernel-checked: no proof assistant is involved.
- Anything about the four-colour theorem, which is cited, not checked.

## Reproduce or challenge it

To reproduce the laboratory's run, build the `math-env` image from a clone of the repository ([Getting Started](/books/math/29-chapter/)) and run:

```bash
img=$(docker image inspect math-env --format '{{.Id}}')
docker run --rm --network none -e UV_OFFLINE=1 -e MATHENV_IMAGE_ID=$img -v "${PWD}:/source:ro" -v "${PWD}/out:/out" math-env uv run --offline math-engine proof kempe --out /out/proof-kempe
```

This Docker form was tested for `kempe` (R-D08, R-D09). To re-verify the recorded run itself without Docker, `math-engine proof --verify experiments/investigations/p06-kempe/runner-runs/2026-11-13-pinned` (R-V06).

Check the transcription against Heawood's Fig. 18 (Quart. J. Pure Appl. Math.
24, 1890, plate and pp. 337–338): are the 25 regions and 69 boundaries right,
and does any error change the conclusion? Then inspect the computation: is a
Kempe chain defined correctly, and would the control catch a dropped or extra
boundary? Propose how to turn this computation into a kernel-checked
statement, and say what that would and would not add.